# 02. Итоговые таблицы активности

Ноутбук строит четыре итоговые таблицы на основе `data/interim/lastfm_time_features.csv`:

1. `listenings_datalens.csv`
2. `user_daily_activity.csv`
3. `user_weekday_hour_profile.csv`
4. `listening_sessions.csv`

## 1. Импорт библиотек

In [10]:
from pathlib import Path

import pandas as pd

## 2. Пути проекта и выходные файлы

In [11]:
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

INTERIM_DIR   = PROJECT_ROOT / "data" / "interim"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
TIME_FEATURES_PATH = INTERIM_DIR / "lastfm_time_features.csv"

LISTENINGS_DATALENS_PATH       = PROCESSED_DIR / "listenings_datalens.csv"
USER_DAILY_ACTIVITY_PATH       = PROCESSED_DIR / "user_daily_activity.csv"
USER_WEEKDAY_HOUR_PROFILE_PATH = PROCESSED_DIR / "user_weekday_hour_profile.csv"
LISTENING_SESSIONS_PATH        = PROCESSED_DIR / "listening_sessions.csv"

SESSION_GAP_MINUTES = 30

TIME_FEATURES_PATH

PosixPath('/Users/zarmeux/hse/course work 1/course-project-listener-classification/data/interim/lastfm_time_features.csv')

## 3. Константы для календарных признаков

In [12]:
MONTH_NUMBER_TO_RU = dict(
    enumerate([
        "Январь",
        "Февраль",
        "Март",
        "Апрель",
        "Май",
        "Июнь",
        "Июль",
        "Август",
        "Сентябрь",
        "Октябрь",
        "Ноябрь",
        "Декабрь",
    ], 1)
)

WEEKDAY_NUMBER_TO_RU = dict(
    enumerate([
        "Понедельник",
        "Вторник",
        "Среда",
        "Четверг",
        "Пятница",
        "Суббота",
        "Воскресенье",
    ])
)

WEEKEND_TO_RU = {
    False: "Будни",
    True:  "Выходные",
}

## 4. Функции построения итоговых таблиц активности

In [13]:
def get_day_part(hour: int) -> str:
    if 6 <= hour <= 11:
        return "Утро"
    if 12 <= hour <= 17:
        return "День"
    if 18 <= hour <= 23:
        return "Вечер"
    return "Ночь"


def prepare_time_features_df(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    df["datetime"]       = pd.to_datetime(df["datetime"])
    df["date_only"]      = pd.to_datetime(df["date_only"]).dt.normalize()
    df["year"]           = df["year"].astype(int)
    df["month_number"]   = df["month_number"].astype(int)
    df["weekday_number"] = df["weekday_number"].astype(int)
    df["hour"]           = df["hour"].astype(int)
    df["is_weekend"]     = df["is_weekend"].astype("boolean").fillna(False).astype(bool)

    return df


def build_listenings_datalens(df: pd.DataFrame) -> pd.DataFrame:
    columns = [
        "username",
        "datetime",
        "date_only",
        "year",
        "month_number",
        "weekday_number",
        "hour",
        "day_part",
        "is_weekend",
        "month",
        "weekday",
        "weekend",
    ]

    result = (
        df[columns]
        .sort_values(["username", "datetime", "hour"], kind="stable")
        .reset_index(drop=True)
        .copy()
    )
    result.insert(0, "listening_id", range(1, len(result) + 1))
    return result


def build_user_daily_activity(df: pd.DataFrame) -> pd.DataFrame:
    users = sorted(df["username"].unique())
    all_dates = pd.date_range(df["date_only"].min(), df["date_only"].max(), freq="D")

    grid = pd.MultiIndex.from_product(
        [users, all_dates],
        names=["username", "date_only"],
    ).to_frame(index=False)

    aggregated = (
        df.groupby(["username", "date_only"], sort=True)
        .agg(
            plays_count       =("datetime", "size"),
            first_play_time   =("datetime", "min"),
            last_play_time    =("datetime", "max"),
            active_hours_count=("hour", "nunique"),
            morning_plays     =("day_part", lambda s: int((s == "Утро").sum())),
            day_plays         =("day_part", lambda s: int((s == "День").sum())),
            evening_plays     =("day_part", lambda s: int((s == "Вечер").sum())),
            night_plays       =("day_part", lambda s: int((s == "Ночь").sum())),
        )
        .reset_index()
    )

    result = grid.merge(aggregated, on=["username", "date_only"], how="left")
    result["year"]           = result["date_only"].dt.year.astype(int)
    result["month_number"]   = result["date_only"].dt.month.astype(int)
    result["weekday_number"] = result["date_only"].dt.weekday.astype(int)
    result["is_weekend"]     = result["weekday_number"].isin([5, 6])
    result["month"]          = result["month_number"].map(MONTH_NUMBER_TO_RU)
    result["weekday"]        = result["weekday_number"].map(WEEKDAY_NUMBER_TO_RU)
    result["weekend"]        = result["is_weekend"].map(WEEKEND_TO_RU)

    int_columns = [
        "plays_count",
        "active_hours_count",
        "morning_plays",
        "day_plays",
        "evening_plays",
        "night_plays",
    ]
    result[int_columns] = result[int_columns].fillna(0).astype(int)
    result["has_activity"] = result["plays_count"].gt(0)

    result["first_play_time"] = result["first_play_time"].dt.strftime("%H:%M:%S")
    result["last_play_time"] = result["last_play_time"].dt.strftime("%H:%M:%S")

    ordered_columns = [
        "username",
        "date_only",
        "year",
        "month_number",
        "weekday_number",
        "is_weekend",
        "has_activity",
        "plays_count",
        "first_play_time",
        "last_play_time",
        "active_hours_count",
        "morning_plays",
        "day_plays",
        "evening_plays",
        "night_plays",
        "month",
        "weekday",
        "weekend",
    ]
    return result[ordered_columns].sort_values(["username", "date_only"], kind="stable").reset_index(drop=True)


def build_user_weekday_hour_profile(df: pd.DataFrame) -> pd.DataFrame:
    users = sorted(df["username"].unique())
    weekdays = list(range(7))
    hours = list(range(24))

    grid = pd.MultiIndex.from_product(
        [users, weekdays, hours],
        names=["username", "weekday_number", "hour"],
    ).to_frame(index=False)

    grid["day_part"]   = grid["hour"].map(get_day_part)
    grid["is_weekend"] = grid["weekday_number"].isin([5, 6])
    grid["weekday"]    = grid["weekday_number"].map(WEEKDAY_NUMBER_TO_RU)
    grid["weekend"]    = grid["is_weekend"].map(WEEKEND_TO_RU)

    aggregated = (
        df.groupby(["username", "weekday_number", "hour"], sort=True)
        .size()
        .reset_index(name="plays_count")
    )

    result = grid.merge(aggregated, on=["username", "weekday_number", "hour"], how="left")
    result["plays_count"] = result["plays_count"].fillna(0).astype(int)

    total_plays = df.groupby("username").size().rename("total_plays")
    result = result.merge(total_plays, on="username", how="left")
    result["plays_share_user"] = (result["plays_count"] / result["total_plays"]).round(6)
    result = result.drop(columns=["total_plays"])

    ordered_columns = [
        "username",
        "weekday_number",
        "hour",
        "day_part",
        "is_weekend",
        "plays_count",
        "plays_share_user",
        "weekday",
        "weekend",
    ]
    return (
        result[ordered_columns]
            .sort_values(["username", "weekday_number", "hour"], kind="stable")
            .reset_index(drop=True)
    )


def build_listening_sessions(df: pd.DataFrame, session_gap_minutes: int = 30) -> pd.DataFrame:
    sorted_df = df.sort_values(["username", "datetime"], kind="stable").copy()
    sorted_df["previous_datetime"] = sorted_df.groupby("username")["datetime"].shift(1)
    sorted_df["time_gap"] = sorted_df["datetime"] - sorted_df["previous_datetime"]
    sorted_df["is_new_session"] = (
        sorted_df["previous_datetime"].isna()
        | sorted_df["time_gap"].gt(pd.Timedelta(minutes=session_gap_minutes))
    )
    sorted_df["session_number"] = (
        sorted_df
            .groupby("username")["is_new_session"]
            .cumsum()
            .astype(int)
    )

    sessions = (
        sorted_df.groupby(["username", "session_number"], sort=True)
        .agg(
            session_start     =("datetime", "min"),
            session_end       =("datetime", "max"),
            plays_count       =("datetime", "size"),
            active_hours_count=("hour", "nunique"),
        )
        .reset_index()
    )

    sessions = sessions.sort_values(["username", "session_number"], kind="stable").reset_index(drop=True)
    sessions.insert(0, "session_id", range(1, len(sessions) + 1))

    sessions["session_date"]     = sessions["session_start"].dt.normalize()
    sessions["year"]             = sessions["session_start"].dt.year.astype(int)
    sessions["month_number"]     = sessions["session_start"].dt.month.astype(int)
    sessions["weekday_number"]   = sessions["session_start"].dt.weekday.astype(int)
    sessions["start_hour"]       = sessions["session_start"].dt.hour.astype(int)
    sessions["end_hour"]         = sessions["session_end"].dt.hour.astype(int)
    sessions["day_part_start"]   = sessions["start_hour"].map(get_day_part)
    sessions["is_weekend"]       = sessions["weekday_number"].isin([5, 6])
    sessions["month"]            = sessions["month_number"].map(MONTH_NUMBER_TO_RU)
    sessions["weekday"]          = sessions["weekday_number"].map(WEEKDAY_NUMBER_TO_RU)
    sessions["weekend"]          = sessions["is_weekend"].map(WEEKEND_TO_RU)
    sessions["duration_minutes"] = (
        (sessions["session_end"] - sessions["session_start"])
        .dt.total_seconds()
        .div(60)
        .round()
        .astype(int)
    )

    ordered_columns = [
        "session_id",
        "session_number",
        "username",
        "session_start",
        "session_end",
        "session_date",
        "year",
        "month_number",
        "weekday_number",
        "start_hour",
        "end_hour",
        "day_part_start",
        "is_weekend",
        "plays_count",
        "duration_minutes",
        "active_hours_count",
        "month",
        "weekday",
        "weekend",
    ]
    return sessions[ordered_columns]

## 5. Загрузка входной таблицы

In [14]:
time_features_df = pd.read_csv(TIME_FEATURES_PATH)
time_features_df = prepare_time_features_df(time_features_df)
print(f"Input shape: {time_features_df.shape}")

Input shape: (166153, 15)


## 6. Построение итоговых таблиц

In [15]:
listenings_datalens_df = build_listenings_datalens(time_features_df)
user_daily_activity_df = build_user_daily_activity(time_features_df)
user_weekday_hour_profile_df = build_user_weekday_hour_profile(time_features_df)
listening_sessions_df = build_listening_sessions(
    time_features_df,
    session_gap_minutes=SESSION_GAP_MINUTES,
)

print(f"listenings_datalens:       {listenings_datalens_df.shape}")
print(f"user_daily_activity:       {user_daily_activity_df.shape}")
print(f"user_weekday_hour_profile: {user_weekday_hour_profile_df.shape}")
print(f"listening_sessions:        {listening_sessions_df.shape}")

listenings_datalens:       (166153, 13)
user_daily_activity:       (341, 18)
user_weekday_hour_profile: (1848, 9)
listening_sessions:        (1465, 19)


## 7. Проверка структуры результатов

In [16]:
assert not listenings_datalens_df.empty
assert listenings_datalens_df["listening_id"].is_unique
assert user_daily_activity_df.groupby("username")["date_only"].size().nunique() == 1
assert user_weekday_hour_profile_df.groupby("username").size().eq(168).all()
assert listening_sessions_df["session_id"].is_unique
assert set(listenings_datalens_df["day_part"].unique()) <= {"Ночь", "Утро", "День", "Вечер"}
assert set(user_daily_activity_df["weekend"].unique()) <= {"Будни", "Выходные"}


## 8. Сохранение файлов

In [17]:
listenings_datalens_df.to_csv(LISTENINGS_DATALENS_PATH, index=False)
user_daily_activity_df.to_csv(USER_DAILY_ACTIVITY_PATH, index=False)
user_weekday_hour_profile_df.to_csv(USER_WEEKDAY_HOUR_PROFILE_PATH, index=False)
listening_sessions_df.to_csv(LISTENING_SESSIONS_PATH, index=False)

print(f"Saved file: {LISTENINGS_DATALENS_PATH}")
print(f"Saved file: {USER_DAILY_ACTIVITY_PATH}")
print(f"Saved file: {USER_WEEKDAY_HOUR_PROFILE_PATH}")
print(f"Saved file: {LISTENING_SESSIONS_PATH}")

Saved file: /Users/zarmeux/hse/course work 1/course-project-listener-classification/data/processed/listenings_datalens.csv
Saved file: /Users/zarmeux/hse/course work 1/course-project-listener-classification/data/processed/user_daily_activity.csv
Saved file: /Users/zarmeux/hse/course work 1/course-project-listener-classification/data/processed/user_weekday_hour_profile.csv
Saved file: /Users/zarmeux/hse/course work 1/course-project-listener-classification/data/processed/listening_sessions.csv


## 9. Предварительный просмотр таблицы прослушиваний

In [18]:
listenings_datalens_df.head()

,listening_id,username,datetime,date_only,year,month_number,weekday_number,hour,day_part,is_weekend,month,weekday,weekend
0,1,Babs_05,2021-01-01 00:02:00,2021-01-01,2021,1,4,0,Ночь,False,Январь,Пятница,Будни
1,2,Babs_05,2021-01-01 00:09:00,2021-01-01,2021,1,4,0,Ночь,False,Январь,Пятница,Будни
2,3,Babs_05,2021-01-01 00:14:00,2021-01-01,2021,1,4,0,Ночь,False,Январь,Пятница,Будни
3,4,Babs_05,2021-01-01 00:19:00,2021-01-01,2021,1,4,0,Ночь,False,Январь,Пятница,Будни
4,5,Babs_05,2021-01-01 00:23:00,2021-01-01,2021,1,4,0,Ночь,False,Январь,Пятница,Будни
